# 🧪 The Prompt Optimization Loop

Use existing eval data to **systematically improve your Canopy summarization prompt**.

```
Register Prompt  →  Run Queries  →  Attach Feedback
       ↑                                    ↓
  New Version  ←  Optimize  ←  Evaluate  ←  Build Dataset
```

**What you'll do here:**
1. **Build** a combined training dataset from the MLflow `eval` dataset (test workspace) and `summary_tests.yaml`
2. **Load and Test** the latest Summarization prompt
3. **Optimize** let MLflow's GEPA automatically rewrite the prompt based on scoring failures
4. **Compare** v1 vs optimized side-by-side and decide whether to promote (this happens after the notebook inside MLflow)

---
## 0. Setup

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import yaml
from typing import Literal

import mlflow
from mlflow import MlflowClient
from mlflow.genai.scorers import scorer
from mlflow.genai.optimize import GepaPromptOptimizer
from mlflow.genai.judges import make_judge
from openai import OpenAI
import pandas as pd

> ⚠️ **Note:** Update the variables below to match your cluster and model endpoint.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# CONFIGURATION — update these to match your environment
# ─────────────────────────────────────────────────────────────────

# MLflow server
MLFLOW_TRACKING_URI = "https://mlflow.redhat-ods-applications.svc.cluster.local:8443"

os.environ["MLFLOW_TRACKING_AUTH"] = "kubernetes"
os.environ["MLFLOW_TRACKING_INSECURE_TLS"] = "true"

NAMESPACE_PATH = "/run/secrets/kubernetes.io/serviceaccount/namespace"
if os.path.exists(NAMESPACE_PATH):
    with open(NAMESPACE_PATH) as f:
        username = f.read().strip().split("-")[0]
        os.environ["MLFLOW_WORKSPACE"] = f"{username}-toolings"

SA_TOKEN_PATH = "/run/secrets/kubernetes.io/serviceaccount/token"
if os.path.exists(SA_TOKEN_PATH):
    with open(SA_TOKEN_PATH) as f:
        os.environ["MLFLOW_TRACKING_TOKEN"] = f.read().strip()

TOOLINGS_WORKSPACE = os.environ.get("MLFLOW_WORKSPACE", "")
TEST_WORKSPACE     = TOOLINGS_WORKSPACE.replace("-toolings", "-test")
print(f"Toolings workspace: {TOOLINGS_WORKSPACE}")
print(f"Test workspace:     {TEST_WORKSPACE}")

# MLflow names
EXPERIMENT_NAME = "summarization"
PROMPT_NAME     = "summarization"

# LLM — point at the same model Canopy uses
LLM_API_KEY  = "fake"
LLM_BASE_URL = "http://llama-32-predictor.ai501.svc.cluster.local:8080/v1"
LLM_MODEL    = "llama32"
if LLM_BASE_URL:
    os.environ["OPENAI_BASE_URL"] = LLM_BASE_URL
os.environ["OPENAI_API_KEY"] = LLM_API_KEY

# Paths to the summarization test config (update if mounted elsewhere)
YAML_PATH         = "../../evals/summarization/summary_tests.yaml"
JUDGE_PROMPT_PATH = "../../evals/summarization/judge_prompt.txt"

In [ ]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)
print(f"✓ Connected to MLflow — workspace: {TOOLINGS_WORKSPACE}, experiment: {EXPERIMENT_NAME}")

mlflow.openai.autolog()
llm_client = OpenAI(
    api_key=LLM_API_KEY,
    **(dict(base_url=LLM_BASE_URL) if LLM_BASE_URL else {}),
)

def _switch_workspace(workspace):
    os.environ["MLFLOW_WORKSPACE"] = workspace
    mlflow.set_tracking_uri("")
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

# Resolve the latest prompt version (prompt lives in toolings)
existing_prompt = mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}@latest")
PROMPT_V1_URI = f"prompts:/{PROMPT_NAME}/{existing_prompt.version}"
print(f"✓ Using prompt '{PROMPT_NAME}' v{existing_prompt.version}  ({PROMPT_V1_URI})")
print(f"\n── Prompt template ──\n{existing_prompt.template}\n─────────────────────")

---
## Step 2: Build the Combined Evaluation Dataset

We combine two sources to create a richer training set for the optimizer:

1. **MLflow `eval` dataset** — records already stored in the `summarization` experiment of the current workspace, accumulated from previous pipeline runs.
2. **`summary_tests.yaml`** — the structured test cases from the prompts repository.

In [ ]:
# Load existing eval dataset from the test workspace
_switch_workspace(TEST_WORKSPACE)
try:
    mlflow_dataset = mlflow.genai.get_dataset(name="eval")
    mlflow_records = mlflow_dataset.to_dict().get("records", [])
    print(f"✓ Loaded MLflow 'eval' dataset from {TEST_WORKSPACE}: {len(mlflow_records)} record(s)")
    for r in mlflow_records[:2]:
        msgs = r.get("inputs", {}).get("messages", [])
        preview = msgs[0]["content"][:70] if msgs else str(r.get("inputs", ""))[:70]
        print(f"  - {preview}...")
except Exception as e:
    print(f"⚠ Could not load 'eval' dataset: {e}")
    mlflow_records = []
finally:
    _switch_workspace(TOOLINGS_WORKSPACE)

In [ ]:
# Load test cases from summary_tests.yaml
with open(YAML_PATH) as f:
    test_config = yaml.safe_load(f)

yaml_records = []
for test in test_config.get("tests", []):
    # Strip session_id — it's a backend routing key, not relevant to the judge
    inputs = {k: v for k, v in test["inputs"].items() if k != "session_id"}
    yaml_records.append({
        "inputs":       inputs,
        "expectations": test.get("expectations", {}),
    })

print(f"✓ Loaded {len(yaml_records)} record(s) from summary_tests.yaml")
for r in yaml_records:
    msgs = r["inputs"].get("messages", [])
    preview = msgs[0]["content"][:70] if msgs else ""
    print(f"  - {preview}...")

In [ ]:
# Combine both sources — keep only inputs + expectations from each record
combined_records = []
for r in mlflow_records:
    combined_records.append({
        "inputs":       r.get("inputs", {}),
        "expectations": r.get("expectations", {}),
    })
combined_records.extend(yaml_records)

print(f"✓ Combined training data: {len(combined_records)} record(s) total")
print(f"  ({len(mlflow_records)} from MLflow eval + {len(yaml_records)} from YAML)")

---
## Step 3: Define the Scorer

We reuse the same `summary_quality` LLM judge that the eval pipeline uses —
loaded from `judge_prompt.txt` alongside the test configuration.

The judge receives the input messages, the generated summary, and the `expected_result`
reference, then answers **"yes"** (good summary) or **"no"** (fails a quality criterion).

In [ ]:
with open(JUDGE_PROMPT_PATH) as f:
    judge_instructions = f.read()

summary_quality = make_judge(
    name="summary_quality",
    instructions=judge_instructions,
    feedback_value_type=Literal["yes", "no"],
    model=f"openai:/{LLM_MODEL}",
    base_url=LLM_BASE_URL + "/chat/completions",
    extra_headers={"Authorization": "Bearer no-key-required"},
)

@scorer
def is_shorter(outputs: str, inputs: dict) -> bool:
    """Is the summary shorter than the input text?"""
    user_content = " ".join(
        m.get("content", "") for m in inputs.get("messages", []) if m.get("role") == "user"
    )
    return len(outputs) < len(user_content)

print("✓ Scorers defined: summary_quality, is_shorter")
print(f"  Judge model: openai:/{LLM_MODEL}")
print(f"  Judge prompt: {JUDGE_PROMPT_PATH}")

---
## Step 4: Baseline Evaluation

Run the current prompt against the combined dataset before optimization, so we have a score to compare against once the automatic pipeline picks up the new prompt version.

In [ ]:
def predict_v1(messages, **kwargs) -> str:
    prompt = mlflow.genai.load_prompt(PROMPT_V1_URI)
    user_content = " ".join(m["content"] for m in messages if m.get("role") == "user")
    response = llm_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": prompt.template},
            {"role": "user",   "content": user_content},
        ],
        temperature=0.0,
    )
    return response.choices[0].message.content


print(f"Running baseline evaluation with prompt v{existing_prompt.version}...")
with mlflow.start_run(run_name="summary_tests_baseline"):
    eval_baseline = mlflow.genai.evaluate(
        data=combined_records,
        predict_fn=predict_v1,
        scorers=[summary_quality, is_shorter],
    )

print("\n── Baseline scores ──")
for metric, value in sorted(eval_baseline.metrics.items()):
    print(f"  {metric:<35} {f'{value:.0%}' if isinstance(value, float) else value}")

---
## Step 5: Optimize with GEPA

**GEPA (Generate, Evaluate, Predict, Adapt)** is MLflow's automated prompt optimizer.

How it works:
1. Runs `predict_fn` on the training data and scores each output with `summary_quality`
2. Uses a *reflection model* to analyse which examples failed and why
3. Generates a rewritten prompt that addresses those weaknesses
4. Registers the improved prompt as a new version in the Prompt Registry

> ⏱️ This takes a few minutes — the optimizer runs multiple evaluation passes internally.

Note: If it times out, try changing `max_metric_calls`

In [ ]:
%pip install -q 'gepa>=0.0.26'

In [ ]:
# Disable autologging before GEPA: autolog captures full OpenAI API traces which
# GEPA includes verbatim in its reflection prompt, easily exceeding the model's context window.
mlflow.openai.autolog(disable=True)


def predict_fn(messages, **kwargs) -> str:
    """
    Predict function for the optimizer.
    mlflow.genai.load_prompt() must be called here so the optimizer
    can intercept the call and identify which template to rewrite.
    """
    prompt = mlflow.genai.load_prompt(PROMPT_V1_URI)  # ← optimizer hooks in here
    user_content = " ".join(m["content"] for m in messages if m.get("role") == "user")
    response = llm_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": prompt.template},
            {"role": "user",   "content": user_content},
        ],
        temperature=0.0,
    )
    return response.choices[0].message.content


print(f"Optimizing '{PROMPT_NAME}' using {LLM_MODEL} as the reflection model...")
print("This may take a few minutes as the optimizer runs multiple passes.\n")

# max_metric_calls controls how many total scorer evaluations GEPA performs (default: 100).
# Lower values finish faster; raise it if you want the optimizer to explore more candidates.
opt_result = mlflow.genai.optimize_prompts(
    predict_fn=predict_fn,
    train_data=combined_records,
    prompt_uris=[PROMPT_V1_URI],
    optimizer=GepaPromptOptimizer(reflection_model=f"openai:/{LLM_MODEL}", max_metric_calls=20),
    scorers=[summary_quality, is_shorter],
)

OPTIMIZED_PROMPT_URI = opt_result.optimized_prompts[0].uri
optimized_version    = opt_result.optimized_prompts[0].version

In [ ]:
print(f"\n✓ Optimization complete!")
print(f"  → New version: v{optimized_version}")

optimized_prompt = mlflow.genai.load_prompt(OPTIMIZED_PROMPT_URI)
print(f"\nOptimized prompt:\n{'─'*50}")
print(optimized_prompt.template)
print("─" * 50)

---
## What's Next?

If the optimized prompt scores better, promote it to test environment and then perhaps to production Canopy! Check Prompts under <USER_NAME>-canopy environment and see the optimized prompt! 

And if you are happy with it, feel free to move it to the <USER_NAME>-tooling environment!

Either way, your feedback loop is now fully closed: real user signals → eval dataset → automated optimization → better Canopy.